In [1]:
import sqlite3
import pandas as pd
import nfl_data_py as nfl

# Connect to the same database file you already created
conn = sqlite3.connect('../data/nfl.db')
cursor = conn.cursor()

# Pull team reference data
teams = nfl.import_team_desc()
print(teams.shape)
print(teams.columns.tolist())

(36, 16)
['team_abbr', 'team_name', 'team_id', 'team_nick', 'team_conf', 'team_division', 'team_color', 'team_color2', 'team_color3', 'team_color4', 'team_logo_wikipedia', 'team_logo_espn', 'team_wordmark', 'team_conference_logo', 'team_league_logo', 'team_logo_squared']


In [2]:
# Testing if team_ids are same for teams that have changed names since 2015 
# Team _id are same for all teams reglardless of historical team name

teams[['team_abbr', 'team_name', 'team_conf', 'team_division', 'team_id']]
teams[teams['team_abbr'].isin(['OAK', 'LV', 'SD', 'LAC', 'STL', 'LAR', 'LA'])][['team_abbr', 'team_name', 'team_id']]

,team_abbr,team_name,team_id
16,LA,Los Angeles Rams,2510
17,LAC,Los Angeles Chargers,4400
18,LAR,Los Angeles Rams,2510
19,LV,Las Vegas Raiders,2520
26,OAK,Oakland Raiders,2520
29,SD,San Diego Chargers,4400
32,STL,St. Louis Rams,2510


In [3]:
## Grabbing team data

cursor.execute('''
CREATE TABLE IF NOT EXISTS teams (
    team_abbr TEXT PRIMARY KEY,
    team_name TEXT,
    team_id TEXT,
    team_nick TEXT,
    team_conf TEXT,
    team_division TEXT,
    team_color TEXT,
    team_color2 TEXT,
    team_logo_espn TEXT,
    team_wordmark TEXT
)
''')

teams_df = teams[['team_abbr', 'team_name', 'team_id', 'team_nick', 
                   'team_conf', 'team_division', 'team_color', 'team_color2', 
                   'team_logo_espn', 'team_wordmark']]
teams_df.to_sql('teams', conn, if_exists='replace', index=False)

conn.commit()
print("Teams table created and loaded!")
print(f"Rows: {len(teams_df)}")

Teams table created and loaded!
Rows: 36


In [4]:
# Grabbing schedule data 

# Pull schedules for 2015 through 2026 (2026 games included but scores will be NULL)
schedules = nfl.import_schedules(range(2015, 2027))
print(schedules.shape)
print(schedules['season'].unique())

# Create games table
cursor.execute('''
CREATE TABLE IF NOT EXISTS games (
    game_id TEXT PRIMARY KEY,
    season INTEGER,
    game_type TEXT,
    week INTEGER,
    gameday TEXT,
    weekday TEXT,
    gametime TEXT,
    home_team TEXT,
    away_team TEXT,
    home_score REAL,
    away_score REAL,
    result REAL,
    total REAL,
    overtime REAL,
    location TEXT,
    home_rest INTEGER,
    away_rest INTEGER,
    div_game INTEGER,
    roof TEXT,
    surface TEXT,
    temp REAL,
    wind REAL,
    referee TEXT,
    stadium_id TEXT,
    stadium TEXT
)
''')

games_df = schedules[[
    'game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime',
    'home_team', 'away_team', 'home_score', 'away_score', 'result', 'total',
    'overtime', 'location', 'home_rest', 'away_rest', 'div_game',
    'roof', 'surface', 'temp', 'wind', 'referee', 'stadium_id', 'stadium'
]]
games_df.to_sql('games', conn, if_exists='replace', index=False)

# Create betting_lines table
cursor.execute('''
CREATE TABLE IF NOT EXISTS betting_lines (
    game_id TEXT PRIMARY KEY,
    spread_line REAL,
    home_spread_odds REAL,
    away_spread_odds REAL,
    total_line REAL,
    over_odds REAL,
    under_odds REAL,
    home_moneyline REAL,
    away_moneyline REAL,
    FOREIGN KEY (game_id) REFERENCES games (game_id)
)
''')

betting_df = schedules[[
    'game_id', 'spread_line', 'home_spread_odds', 'away_spread_odds',
    'total_line', 'over_odds', 'under_odds', 'home_moneyline', 'away_moneyline'
]]
betting_df.to_sql('betting_lines', conn, if_exists='replace', index=False)

conn.commit()
print("Games and betting_lines tables loaded!")
print(f"Games: {len(games_df)}, Betting lines: {len(betting_df)}")

(3300, 46)
[2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026]
Games and betting_lines tables loaded!
Games: 3300, Betting lines: 3300


In [5]:
## Grabbing players

players = nfl.import_players()
print(players.shape)

cursor.execute('''
CREATE TABLE IF NOT EXISTS players (
    gsis_id TEXT PRIMARY KEY,
    display_name TEXT,
    first_name TEXT,
    last_name TEXT,
    position TEXT,
    position_group TEXT,
    height REAL,
    weight REAL,
    college_name TEXT,
    birth_date TEXT,
    rookie_season INTEGER,
    last_season INTEGER,
    latest_team TEXT,
    status TEXT,
    years_of_experience INTEGER,
    draft_year REAL,
    draft_round REAL,
    draft_pick REAL,
    draft_team TEXT
)
''')

players_df = players[[
    'gsis_id', 'display_name', 'first_name', 'last_name', 'position', 'position_group',
    'height', 'weight', 'college_name', 'birth_date', 'rookie_season', 'last_season',
    'latest_team', 'status', 'years_of_experience', 'draft_year', 'draft_round',
    'draft_pick', 'draft_team'
]]
players_df.to_sql('players', conn, if_exists='replace', index=False)

conn.commit()
print("Players table loaded!")
print(f"Players: {len(players_df)}")

(25037, 39)
Players table loaded!
Players: 25037


In [6]:
# grabbing weekly stats

weekly = nfl.import_weekly_data(range(2015, 2025))
print(weekly.shape)

cursor.execute('''
CREATE TABLE IF NOT EXISTS player_game_stats (
    player_id TEXT,
    player_name TEXT,
    position TEXT,
    recent_team TEXT,
    season INTEGER,
    week INTEGER,
    season_type TEXT,
    opponent_team TEXT,
    completions INTEGER,
    attempts INTEGER,
    passing_yards REAL,
    passing_tds INTEGER,
    interceptions REAL,
    sacks REAL,
    passing_first_downs REAL,
    passing_epa REAL,
    sack_fumbles_lost INTEGER,
    carries INTEGER,
    rushing_yards REAL,
    rushing_tds INTEGER,
    rushing_first_downs REAL,
    rushing_epa REAL,
    rushing_fumbles_lost REAL,
    receptions INTEGER,
    targets INTEGER,
    receiving_yards REAL,
    receiving_tds INTEGER,
    receiving_first_downs REAL,
    receiving_epa REAL,
    receiving_fumbles_lost REAL,
    target_share REAL,
    air_yards_share REAL,
    fantasy_points REAL,
    fantasy_points_ppr REAL,
    PRIMARY KEY (player_id, season, week)
)
''')

player_stats_df = weekly[[
    'player_id', 'player_display_name', 'position', 'recent_team', 'season', 'week',
    'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards',
    'passing_tds', 'interceptions', 'sacks', 'passing_first_downs', 'passing_epa',
    'sack_fumbles_lost', 'carries', 'rushing_yards', 'rushing_tds',
    'rushing_first_downs', 'rushing_epa', 'rushing_fumbles_lost',
    'receptions', 'targets', 'receiving_yards', 'receiving_tds',
    'receiving_first_downs', 'receiving_epa', 'receiving_fumbles_lost',
    'target_share', 'air_yards_share', 'fantasy_points', 'fantasy_points_ppr'
]].rename(columns={'player_display_name': 'player_name'})

player_stats_df.to_sql('player_game_stats', conn, if_exists='replace', index=False)

conn.commit()
print("Player game stats table loaded!")
print(f"Rows: {len(player_stats_df)}")

Downcasting floats.
(54479, 53)
Player game stats table loaded!
Rows: 54479


In [7]:
# grabbing season data

seasonal = nfl.import_seasonal_data(range(2015, 2025))
print(seasonal.shape)

cursor.execute('''
CREATE TABLE IF NOT EXISTS player_season_stats (
    player_id TEXT,
    season INTEGER,
    season_type TEXT,
    games INTEGER,
    completions INTEGER,
    attempts INTEGER,
    passing_yards REAL,
    passing_tds INTEGER,
    interceptions REAL,
    passing_first_downs REAL,
    passing_epa REAL,
    sack_fumbles_lost INTEGER,
    carries INTEGER,
    rushing_yards REAL,
    rushing_tds INTEGER,
    rushing_first_downs REAL,
    rushing_epa REAL,
    rushing_fumbles_lost REAL,
    receptions INTEGER,
    targets INTEGER,
    receiving_yards REAL,
    receiving_tds INTEGER,
    receiving_first_downs REAL,
    receiving_epa REAL,
    receiving_fumbles_lost REAL,
    target_share REAL,
    air_yards_share REAL,
    fantasy_points REAL,
    fantasy_points_ppr REAL,
    PRIMARY KEY (player_id, season)
)
''')

season_stats_df = seasonal[[
    'player_id', 'season', 'season_type', 'games', 'completions', 'attempts',
    'passing_yards', 'passing_tds', 'interceptions', 'passing_first_downs', 'passing_epa',
    'sack_fumbles_lost', 'carries', 'rushing_yards', 'rushing_tds',
    'rushing_first_downs', 'rushing_epa', 'rushing_fumbles_lost',
    'receptions', 'targets', 'receiving_yards', 'receiving_tds',
    'receiving_first_downs', 'receiving_epa', 'receiving_fumbles_lost',
    'tgt_sh', 'ay_sh', 'fantasy_points', 'fantasy_points_ppr'
]].rename(columns={'tgt_sh': 'target_share', 'ay_sh': 'air_yards_share'})

season_stats_df.to_sql('player_season_stats', conn, if_exists='replace', index=False)

conn.commit()
print("Player season stats table loaded!")
print(f"Rows: {len(season_stats_df)}")

(6098, 58)
Player season stats table loaded!
Rows: 6098


In [ ]:

# Build depth_charts_historical (2015-2025)

# 1. Pull the old-format historical data (2015-2024)
depth_charts_historical = nfl.import_depth_charts(range(2015, 2025))

# 2. Pull the 2025 snapshot-format data separately
depth_2025 = nfl.import_depth_charts([2025])

# 3. Get each week's earliest game date for 2025, to match snapshots to weeks
season_2025_weeks = pd.read_sql_query('''
    SELECT DISTINCT season, week, gameday 
    FROM games 
    WHERE season = 2025 AND game_type = 'REG'
    ORDER BY week
''', conn)

week_starts = season_2025_weeks.groupby('week')['gameday'].min().reset_index()
week_starts['gameday'] = pd.to_datetime(week_starts['gameday'])

# 4. Match each week to the closest snapshot taken before that week started
depth_2025 = depth_2025.copy()
depth_2025['dt_parsed'] = pd.to_datetime(depth_2025['dt']).dt.tz_localize(None)

matched_rows = []
for _, row in week_starts.iterrows():
    week_num = row['week']
    week_start_date = row['gameday']
    valid_snapshots = depth_2025[depth_2025['dt_parsed'] < week_start_date]
    if len(valid_snapshots) == 0:
        print(f"Week {week_num}: no snapshot found before {week_start_date}")
        continue
    latest_valid_dt = valid_snapshots['dt_parsed'].max()
    week_snapshot = depth_2025[depth_2025['dt_parsed'] == latest_valid_dt].copy()
    week_snapshot['week'] = week_num
    week_snapshot['season'] = 2025
    matched_rows.append(week_snapshot)

depth_2025_weekly = pd.concat(matched_rows, ignore_index=True)

# 5. Reformat 2025 data to match the historical schema
depth_2025_formatted = depth_2025_weekly[[
    'season', 'team', 'week', 'gsis_id', 'player_name', 'pos_abb', 'pos_name'
]].rename(columns={
    'team': 'club_code',
    'player_name': 'full_name',
    'pos_abb': 'position',
    'pos_name': 'depth_position'
})
depth_2025_formatted['game_type'] = 'REG'
depth_2025_formatted['depth_team'] = None
depth_2025_formatted['formation'] = None
depth_2025_formatted['jersey_number'] = None
depth_2025_formatted = depth_2025_formatted[[
    'season', 'club_code', 'week', 'game_type', 'depth_team', 'gsis_id',
    'full_name', 'position', 'depth_position', 'formation', 'jersey_number'
]]

# 6. Create the table
cursor.execute('''
CREATE TABLE IF NOT EXISTS depth_charts_historical (
    season INTEGER,
    club_code TEXT,
    week INTEGER,
    game_type TEXT,
    depth_team TEXT,
    gsis_id TEXT,
    full_name TEXT,
    position TEXT,
    depth_position TEXT,
    formation TEXT,
    jersey_number TEXT
)
''')

# 7. Load 2015-2024, then append 2025
depth_hist_df = depth_charts_historical[[
    'season', 'club_code', 'week', 'game_type', 'depth_team', 'gsis_id',
    'full_name', 'position', 'depth_position', 'formation', 'jersey_number'
]]
depth_hist_df.to_sql('depth_charts_historical', conn, if_exists='replace', index=False)
depth_2025_formatted.to_sql('depth_charts_historical', conn, if_exists='append', index=False)

conn.commit()

# 8. Sanity check
result = pd.read_sql_query('SELECT season, COUNT(*) as rows FROM depth_charts_historical GROUP BY season', conn)
print("Depth charts historical loaded (2015-2025)!")
print(result)

Depth charts historical loaded (2015-2025)!
    season   rows
0     2015  37058
1     2016  36612
2     2017  36620
3     2018  36560
4     2019  36308
5     2020  36168
6     2021  37487
7     2022  37780
8     2023  37327
9     2024  37312
10    2025  40678


In [ ]:
## updated depth chart formate
# Only pull 2026 this is the live season we want current snapshot from
depth_charts_current = nfl.import_depth_charts([2026])
print(depth_charts_current.shape)

# Filter down to just the single most recent snapshot
latest_dt = depth_charts_current['dt'].max()
print(f"Latest snapshot: {latest_dt}")

depth_charts_current_latest = depth_charts_current[depth_charts_current['dt'] == latest_dt]
print(depth_charts_current_latest.shape)

# Rebuild the table with just this clean snapshot
cursor.execute('DROP TABLE IF EXISTS depth_charts_current')

cursor.execute('''
CREATE TABLE IF NOT EXISTS depth_charts_current (
    dt TEXT,
    team TEXT,
    player_name TEXT,
    gsis_id TEXT,
    pos_grp TEXT,
    pos_name TEXT,
    pos_abb TEXT,
    pos_slot REAL,
    pos_rank REAL
)
''')

depth_current_df = depth_charts_current_latest[[
    'dt', 'team', 'player_name', 'gsis_id', 'pos_grp', 'pos_name', 'pos_abb',
    'pos_slot', 'pos_rank'
]]
depth_current_df.to_sql('depth_charts_current', conn, if_exists='replace', index=False)

conn.commit()
print("Depth charts current (2026 only, latest snapshot) loaded!")
print(f"Rows: {len(depth_current_df)}")

(426595, 12)
Latest snapshot: 2026-08-12T08:11:43Z
(3252, 12)
Depth charts current (2026 only, latest snapshot) loaded!
Rows: 3252


In [10]:
injuries = nfl.import_injuries(range(2015, 2026))
print(injuries.shape)

cursor.execute('''
CREATE TABLE IF NOT EXISTS injuries (
    season INTEGER,
    game_type TEXT,
    team TEXT,
    week INTEGER,
    gsis_id TEXT,
    full_name TEXT,
    position TEXT,
    report_primary_injury TEXT,
    report_status TEXT,
    practice_status TEXT,
    date_modified TEXT
)
''')

injuries_df = injuries[[
    'season', 'game_type', 'team', 'week', 'gsis_id', 'full_name', 'position',
    'report_primary_injury', 'report_status', 'practice_status', 'date_modified'
]]
injuries_df.to_sql('injuries', conn, if_exists='replace', index=False)

conn.commit()
print("Injuries table loaded!")
print(f"Rows: {len(injuries_df)}")

(60788, 17)
Injuries table loaded!
Rows: 60788


In [ ]:
# Final look at how many rows we have

tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

for table in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as rows FROM {table}", conn)
    print(f"{table}: {count['rows'][0]} rows")

                          name
0                   team_games
1   sample_season_player_stats
2          sample_season_games
3    season_simulation_summary
4        season_records_by_sim
5         playoff_seeds_by_sim
6       playoff_results_by_sim
7          all_simulated_games
8           final_season_games
9         final_season_players
10         final_playoff_seeds
11         final_playoff_games
12       final_playoff_players
13              game_personnel
14                       teams
15                       games
16               betting_lines
17                     players
18           player_game_stats
19         player_season_stats
20     depth_charts_historical
21        depth_charts_current
22                    injuries
team_games: 6600 rows
sample_season_player_stats: 4896 rows
sample_season_games: 272 rows
season_simulation_summary: 32 rows
season_records_by_sim: 16000 rows
playoff_seeds_by_sim: 7000 rows
playoff_results_by_sim: 500 rows
all_simulated_games: 136000 rows

In [12]:
# One join test KC week one does all info show?
# Answer: Yes and it matches google this is really cool!!!

test_query = '''
SELECT 
    g.season, g.week, g.home_team, g.away_team,
    p.player_name, p.position, p.receptions, p.targets, p.receiving_yards
FROM games g
JOIN player_game_stats p ON g.season = p.season 
    AND g.week = p.week 
    AND g.home_team = p.recent_team
WHERE g.season = 2024 AND g.week = 1 AND g.home_team = 'KC'
LIMIT 10
'''

result = pd.read_sql_query(test_query, conn)
result

,season,week,home_team,away_team,player_name,position,receptions,targets,receiving_yards
0,2024,1,KC,BAL,Carson Steele,RB,0,0,0.0
1,2024,1,KC,BAL,Isiah Pacheco,RB,2,3,33.0
2,2024,1,KC,BAL,JuJu Smith-Schuster,WR,0,1,0.0
3,2024,1,KC,BAL,Justin Watson,WR,1,1,25.0
4,2024,1,KC,BAL,Noah Gray,TE,3,3,37.0
5,2024,1,KC,BAL,Patrick Mahomes,QB,1,1,2.0
6,2024,1,KC,BAL,Rashee Rice,WR,7,9,103.0
7,2024,1,KC,BAL,Samaje Perine,RB,1,2,10.0
8,2024,1,KC,BAL,Travis Kelce,TE,3,4,34.0
9,2024,1,KC,BAL,Xavier Worthy,WR,2,3,47.0


In [13]:
# Re-create the lookup tables this cell depends on
games_lookup = pd.read_sql_query('''
    SELECT game_id, season, week, home_team, away_team
    FROM games
''', conn)

player_stats = pd.read_sql_query('SELECT * FROM player_game_stats', conn)

# Pull team_id mapping for every abbreviation (including old ones like OAK, SD, STL)
team_id_map = pd.read_sql_query('SELECT team_abbr, team_id FROM teams', conn)

# Attach team_id to the games lookup (for both home and away)
home_lookup = games_lookup[['game_id', 'season', 'week', 'home_team']].rename(columns={'home_team': 'team_abbr'})
away_lookup = games_lookup[['game_id', 'season', 'week', 'away_team']].rename(columns={'away_team': 'team_abbr'})
team_game_lookup = pd.concat([home_lookup, away_lookup], ignore_index=True)

team_game_lookup = team_game_lookup.merge(team_id_map, on='team_abbr', how='left')

# Attach team_id to player_stats too, based on recent_team
player_stats_with_id = player_stats.merge(
    team_id_map.rename(columns={'team_abbr': 'recent_team'}),
    on='recent_team',
    how='left'
)

# Now merge on season + week + team_id, NOT on raw abbreviation
player_stats_with_gameid = player_stats_with_id.merge(
    team_game_lookup[['game_id', 'season', 'week', 'team_id']],
    on=['season', 'week', 'team_id'],
    how='left'
)

print(f"Total rows: {len(player_stats_with_gameid)}")
print(f"Rows with game_id matched: {player_stats_with_gameid['game_id'].notna().sum()}")
print(f"Rows with NO match: {player_stats_with_gameid['game_id'].isna().sum()}")

Total rows: 54479
Rows with game_id matched: 54479
Rows with NO match: 0


In [14]:
# Clean up: keep only the columns we want, drop the helper columns from the merge
final_player_stats = player_stats_with_gameid.drop(columns=['team_id']).copy()

# Overwrite player_game_stats with the version that now includes game_id
cursor.execute("DROP TABLE IF EXISTS player_game_stats")

cursor.execute('''
CREATE TABLE IF NOT EXISTS player_game_stats (
    player_id TEXT,
    game_id TEXT,
    player_name TEXT,
    position TEXT,
    recent_team TEXT,
    season INTEGER,
    week INTEGER,
    season_type TEXT,
    opponent_team TEXT,
    completions INTEGER,
    attempts INTEGER,
    passing_yards REAL,
    passing_tds INTEGER,
    interceptions REAL,
    sacks REAL,
    passing_first_downs REAL,
    passing_epa REAL,
    sack_fumbles_lost INTEGER,
    carries INTEGER,
    rushing_yards REAL,
    rushing_tds INTEGER,
    rushing_first_downs REAL,
    rushing_epa REAL,
    rushing_fumbles_lost REAL,
    receptions INTEGER,
    targets INTEGER,
    receiving_yards REAL,
    receiving_tds INTEGER,
    receiving_first_downs REAL,
    receiving_epa REAL,
    receiving_fumbles_lost REAL,
    target_share REAL,
    air_yards_share REAL,
    fantasy_points REAL,
    fantasy_points_ppr REAL,
    PRIMARY KEY (player_id, season, week)
)
''')

final_player_stats.to_sql('player_game_stats', conn, if_exists='append', index=False)

conn.commit()
print("player_game_stats updated with game_id!")
print(f"Rows: {len(final_player_stats)}")

player_game_stats updated with game_id!
Rows: 54479


In [15]:
check = pd.read_sql_query('''
    SELECT game_id, player_name, passing_yards, receiving_yards 
    FROM player_game_stats 
    LIMIT 5
''', conn)
check

,game_id,player_name,passing_yards,receiving_yards
0,2015_04_JAX_IND,Matt Hasselbeck,282.0,0.0
1,2015_05_IND_HOU,Matt Hasselbeck,213.0,0.0
2,2015_11_IND_ATL,Matt Hasselbeck,213.0,0.0
3,2015_12_TB_IND,Matt Hasselbeck,315.0,0.0
4,2015_13_IND_PIT,Matt Hasselbeck,169.0,0.0


In [16]:
tables = ['teams', 'games', 'betting_lines', 'players', 'player_game_stats', 
          'player_season_stats', 'depth_charts_historical', 'depth_charts_current', 'injuries']

for table in tables:
    print(f"\n--- {table} ---")
    cursor.execute(f"PRAGMA table_info({table})")
    columns = cursor.fetchall()
    for col in columns:
        print(col[1], '-', col[2])


--- teams ---
team_abbr - TEXT
team_name - TEXT
team_id - INTEGER
team_nick - TEXT
team_conf - TEXT
team_division - TEXT
team_color - TEXT
team_color2 - TEXT
team_logo_espn - TEXT
team_wordmark - TEXT

--- games ---
game_id - TEXT
season - INTEGER
game_type - TEXT
week - INTEGER
gameday - TEXT
weekday - TEXT
gametime - TEXT
home_team - TEXT
away_team - TEXT
home_score - REAL
away_score - REAL
result - REAL
total - REAL
overtime - REAL
location - TEXT
home_rest - INTEGER
away_rest - INTEGER
div_game - INTEGER
roof - TEXT
surface - TEXT
temp - REAL
wind - REAL
referee - TEXT
stadium_id - TEXT
stadium - TEXT

--- betting_lines ---
game_id - TEXT
spread_line - REAL
home_spread_odds - REAL
away_spread_odds - REAL
total_line - REAL
over_odds - REAL
under_odds - REAL
home_moneyline - REAL
away_moneyline - REAL

--- players ---
gsis_id - TEXT
display_name - TEXT
first_name - TEXT
last_name - TEXT
position - TEXT
position_group - TEXT
height - REAL
weight - REAL
college_name - TEXT
birth_date 